<a href="https://colab.research.google.com/github/Aqib2607/AI/blob/master/colab/02_drive_mount.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. Google Drive Mounting & API Quota Preflight

Mounts Google Drive persistently at `/content/drive`, authoritatively validates real Google Drive account storage quota via Google Drive API v3 (`drive.about.get`), confirms target folder structure (`AI - Google Drive` ID: `11BdZx7pI2XyEmiJjpZJjTCIX1V41vKhd`), and isolates FUSE diagnostics.

### Authentication & Scope Rules
- **Target Account**: `aqibjawwad2607@gmail.com`
- **Target Folder**: `AI - Google Drive` (Folder ID: `11BdZx7pI2XyEmiJjpZJjTCIX1V41vKhd`)
- **Root Path After Mount**: `/content/drive/MyDrive/AI - Google Drive`
- **Scope Isolation**: Operations are strictly restricted to `/content/drive/MyDrive/AI - Google Drive/GLM-5.2/`.
- **Authoritative Quota**: Discovered at runtime via Google Drive API (FUSE `shutil.disk_usage()` is diagnostic only).

### Step 1: Repository Bootstrap & Dependency Setup

In [ ]:
import os
import sys
import subprocess

# Ensure repository is cloned and synchronized to latest commit in Colab runtime
REPO_DIR = '/content/glm52-drive-runtime'
if not os.path.exists(REPO_DIR):
    print(f"Cloning GLM-5.2 repository into {REPO_DIR}...")
    subprocess.run(['git', 'clone', 'https://github.com/Aqib2607/AI.git', REPO_DIR], check=True)
else:
    print(f"Synchronizing repository at {REPO_DIR} to latest master...")
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'master'], check=False)

# Install dependencies
!pip install -q google-api-python-client google-auth rich tabulate requests

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

DRIVE_CHECK_SCRIPT = os.path.join(REPO_DIR, 'scripts', 'drive_check.py')
if not os.path.exists(DRIVE_CHECK_SCRIPT):
    raise FileNotFoundError(f"Expected health check script at {DRIVE_CHECK_SCRIPT}")
print(f"✓ Verified health check script at: {DRIVE_CHECK_SCRIPT}")

### Step 2: Mount Google Drive & Authenticate Drive API

In [ ]:
import os
import importlib

# 1. Mount Google Drive for account: aqibjawwad2607@gmail.com
try:
    drive = importlib.import_module('google' + '.colab.drive')
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f"Note: Drive mount skipped in local development ({e})")

# 2. Authenticate Google Colab User for Drive API v3 Quota Access
try:
    colab_auth = importlib.import_module('google' + '.colab.auth')
    colab_auth.authenticate_user()
    print("✓ Google Colab API authentication completed")
except Exception as e:
    print(f"Note: Google Colab auth prompt skipped ({e})")

# 3. Initialize project directories inside 'AI - Google Drive'
DRIVE_ROOT = '/content/drive/MyDrive/AI - Google Drive'
BASE_DIR = os.path.join(DRIVE_ROOT, 'GLM-5.2')
MODEL_DIR = os.path.join(BASE_DIR, 'model')
RUNTIME_DIR = os.path.join(BASE_DIR, 'runtime')
LOGS_DIR = os.path.join(BASE_DIR, 'logs')
MANIFESTS_DIR = os.path.join(BASE_DIR, 'manifests')
BENCHMARKS_DIR = os.path.join(BASE_DIR, 'benchmarks')

for d in [MODEL_DIR, RUNTIME_DIR, LOGS_DIR, MANIFESTS_DIR, BENCHMARKS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✓ Target project structure initialized at: {BASE_DIR}")

In [ ]:
%cd /content/glm52-drive-runtime

!git fetch origin master
!git reset --hard origin/master

!git log -1 --oneline

!python scripts/drive_check.py --help

### Step 3: Run Authoritative Drive API Quota & Storage Preflight

In [ ]:
# Execute drive_check.py to query Google Drive API v3 authoritative quota and validate target folder ID
!python /content/glm52-drive-runtime/scripts/drive_check.py \
  --path "/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model" \
  --required-gb 400 \
  --recommended-gb 450 \
  --folder-id "11BdZx7pI2XyEmiJjpZJjTCIX1V41vKhd"